In [ ]:
from thbsplines.hierarchical_space import HierarchicalSpace
#from THBSplines.src.cartesian_mesh import CartesianMesh
import numpy as np
import scipy.sparse as sp
import dolfinx
from mpi4py import MPI
import basix.ufl
import pyvista

import cffi
import numba
import numba.core.typing.cffi_utils as cffi_support
from dolfinx.jit import ffcx_jit
from dolfinx import default_real_type, default_scalar_type, geometry
rtype = default_real_type
dtype = default_scalar_type
import ufl
from ffcx.codegeneration.utils import empty_void_pointer
from ffcx.codegeneration.utils import numba_ufcx_kernel_signature as ufcx_signature

import numpy.typing as npt
from thbsplines.refinement import refine
from thbsplines.fenicsx.mesh import build_mesh
from thbsplines.fenicsx.functionspace import build_dofmap, fill_function_space, create_spline_space
from thbsplines.fenicsx.solvers import solve_problem
from thbsplines.fenicsx.adaptivity import dorfler_marking
from thbsplines.fenicsx.kernels import make_linear_kernel, make_bilinear_kernel
        

In [ ]:
p0 = 2
m=4
n_refinements = 1
knots1 = np.array([-1., 0., 1.], dtype=np.float64)
knots1 = refine(knots1, p=p0, n_times=n_refinements)
# log_initial_mesh_size = np.log2(np.max(np.diff(knots1)))
err_cells = {}
hs = HierarchicalSpace(knots=[knots1, knots1, knots1], degrees=[p0])

In [ ]:
for level, cells in err_cells.items():
    hs.refine(cells, level, refine_neighbours=True, refine_T_neighbours=True, m=m)
disconnected_mesh, thb_operators, N_max, _ = build_mesh(hs=hs, dim3=True)

In [ ]:
# total_active_cells = sum(len(hs.hmesh.aelem_level[l]) for l in range(hs.nlevels))

# all_cells = np.empty((2**hs.dim*total_active_cells, hs.dim), dtype=np.float64) # will have coarser cells on top and finer on bottom
# thb_operators: dict[tuple[int, int], npt.NDArray[np.float32]] = {}
# N_max = 0 # maximum amount of dofs in a cell
# current_idx = 0
# for l in range(hs.nlevels):
#     active_cells_l = hs.hmesh.aelem_level[l]
#     if len(active_cells_l)==0:
#         continue
   
#     thb_operators_list = hs.local_multi_level_extraction_operator3(active_cells_l, l, l)
#     thb_operators.update({(l, cell): op for cell, op in zip(active_cells_l, thb_operators_list)})
#     #identity matrix for B-Splines.
#     print("Computed THB-operators!")
#     if thb_operators_list:
#         level_max = max(op.shape[0] for op in thb_operators_list)
#         N_max = max(N_max, level_max)

#     mesh = CartesianMesh(hs.hmesh.one_d_indices[l], len(hs.hmesh.one_d_indices[l]))
#     my_cells_l = mesh.cells[active_cells_l]

#     n_cells = len(my_cells_l)
#     x_coords = my_cells_l[:, 0, :].astype(np.float64)
#     y_coords = my_cells_l[:, 1, :].astype(np.float64)
#     z_coords = my_cells_l[:, 2, :].astype(np.float64)

#     start, end = current_idx, current_idx+(2**hs.dim*n_cells)

#     view = all_cells[start:end]
#     view[0::8] = np.column_stack((x_coords[:, 0], y_coords[:, 0], z_coords[:, 0]))  # (xmin, ymin, zmin)
#     view[1::8] = np.column_stack((x_coords[:, 1], y_coords[:, 0], z_coords[:, 0]))  # (xmax, ymin, zmin)
#     view[2::8] = np.column_stack((x_coords[:, 0], y_coords[:, 1], z_coords[:, 0]))  # (xmin, ymax, zmin)
#     view[3::8] = np.column_stack((x_coords[:, 1], y_coords[:, 1], z_coords[:, 0]))  # (xmax, ymax, zmin)
#     view[4::8] = np.column_stack((x_coords[:, 0], y_coords[:, 0], z_coords[:, 1]))  # (xmin, ymin, zmax)
#     view[5::8] = np.column_stack((x_coords[:, 1], y_coords[:, 0], z_coords[:, 1]))  # (xmax, ymin, zmax)
#     view[6::8] = np.column_stack((x_coords[:, 0], y_coords[:, 1], z_coords[:, 1]))  # (xmin, ymax, zmax)
#     view[7::8] = np.column_stack((x_coords[:, 1], y_coords[:, 1], z_coords[:, 1]))  # (xmax, ymax, zmax)
#     current_idx=end
# pass
# del mesh # free this big object
# all_cells = np.array(all_cells).reshape(-1, 3)

# coordinates = np.arange(len(all_cells), dtype=np.int32).reshape(-1, 2**hs.dim)
# coordinate_element = basix.ufl.element("Q", "hexahedron", 1, shape=(hs.dim,), dtype=np.float64)
# disconnected_mesh = dolfinx.mesh.create_mesh(MPI.COMM_WORLD, cells=coordinates, e=coordinate_element, x=all_cells)
# del all_cells
# del coordinates
# #del thb_operators_list

In [ ]:
# from dolfinx import io

# --- Your existing mesh creation code ---
# disconnected_mesh = dolfinx.mesh.create_mesh(MPI.COMM_WORLD, cells=coordinates, e=coordinate_element, x=all_cells)

# Export using VTXWriter
#with io.VTXWriter(disconnected_mesh.comm, "mesh_3d.bp", disconnected_mesh, engine="BP4") as vtx:
#    vtx.write(7.0)  # 0.0 represents the time step

In [ ]:
# topology, cell_types, geometry = dolfinx.plot.vtk_mesh(disconnected_mesh)
# grid = pyvista.UnstructuredGrid(topology, cell_types, geometry)

# plotter = pyvista.Plotter()
# plotter.add_mesh(grid.shrink(.95), show_edges=False, color="#03bb85")
# plotter.view_yz()
# plotter.show(jupyter_backend="static")
# print(f"Number of points in PyVista grid: {grid.n_points}")

In [ ]:
legendre_elt = basix.ufl.element(
    "DG",
    "hexahedron",
    degree=p0,
    lagrange_variant=basix.LagrangeVariant.legendre,
    dtype=np.float64
)
V = dolfinx.fem.functionspace(disconnected_mesh, legendre_elt)
print(f"Number of degrees of freedom: {V.dofmap.index_map.size_global}")
dx_custom = ufl.Measure("dx", domain=disconnected_mesh, metadata={"quadrature_degree": 8})
u,v = ufl.TrialFunction(V), ufl.TestFunction(V) 
my_x = ufl.SpatialCoordinate(disconnected_mesh)
#f = dolfinx.fem.Function(V)
#f.interpolate(lambda x: (np.tanh(9*x[1]-9*x[0]+9*x[2])+1)/9. + 1./(1.5*np.exp((10.*x[0]-6.)**2 + (10.*x[1]+7)**2 + (10.*x[2]-0.1)**2))) 
#f.interpolate(lambda x: x[0]**3+1+0.2*x[1]-0.87*x[2]*x[1] + x[1]*x[0]-0.05*x[0]**2*x[2]**2 + 10.*x[0]*x[1]**2*x[2])
#f = my_x[0]*my_x[1] - my_x[1]*my_x[2] + 0.3*my_x[2]**2
#f = (ufl.tanh(9*my_x[1]-9*my_x[0]+9*my_x[2])+1)/9. + 1./(1.5*ufl.exp(ufl.sqrt((10.*my_x[0]-6.)**2 + (10.*my_x[1]+7.)**2 + (10.*my_x[2]-0.1)**2))) 
f = 1./(1.5*ufl.exp(ufl.sqrt((10.*my_x[0]-6.)**2 + (10.*my_x[1]+7.)**2 + (10.*my_x[2]-0.1)**2))) 
#f = 1./(1.5*ufl.exp((10.*my_x[0]-6.)**2 + (10.*my_x[1]+7.)**2 + (10.*my_x[2]-0.1)**2))
#f = my_x[2]*my_x[1]**3+my_x[0]*(my_x[1]-0.7)**2 + 0.3*my_x[2]-0.1
a0 = ufl.inner(u, v) * dx_custom
f0 = ufl.inner(f, v)*dx_custom

f_square_integral = dolfinx.fem.assemble_scalar(dolfinx.fem.form(ufl.inner(f,f)*dx_custom, dtype=np.float64))
f_sq_integral = np.sqrt(disconnected_mesh.comm.allreduce(f_square_integral, op=MPI.SUM))

msh = disconnected_mesh
ufcxa0, _, _ = ffcx_jit(msh.comm, a0, form_compiler_options={"scalar_type": dtype})  # type: ignore
kernela0 = getattr(ufcxa0.form_integrals[0], f"tabulate_tensor_{np.dtype(dtype).name}")  # type: ignore

ufcxf0, _, _ = ffcx_jit(msh.comm, f0, form_compiler_options={"scalar_type": dtype})  # type: ignore
kernelf0 = getattr(ufcxf0.form_integrals[0], f"tabulate_tensor_{np.dtype(dtype).name}")  # type: ignore

# ffi = cffi.FFI()

In [ ]:
dofmap, padded_cells_to_dofs = build_dofmap(hierarchical_space=hs, mesh=disconnected_mesh, N_max=N_max, morton=True)

In [ ]:
M = hs._bezier_to_legendre(degree = p0)
S_indices = np.arange(p0+1, dtype=np.float64)
# scaling for unnormalised Legendre basis polynomials
S_inv = (1./np.sqrt(2.*S_indices+1.))*np.identity(p0+1, dtype=np.float64) # Scaling factor, since fenicsx uses orthonormal legendre polynomials
T = np.asfortranarray(np.kron(np.kron(M, M), M).T @ np.kron(np.kron(S_inv, S_inv), S_inv), dtype=np.float64)
local_dofs_size = T.shape[1]

operator_shape = (N_max, local_dofs_size)
# Create a custom space that holds the content of each matrix for the relevant cell.
# degree 0 because the value is constant over each cell
C_element = basix.ufl.element(
    "DG",
    cell='hexahedron',
    degree=0,
    shape = operator_shape,
    dtype=np.float64
)
C_space = dolfinx.fem.functionspace(disconnected_mesh, C_element)# ("DG", 0, operator_shape, np.float32))
C_func = dolfinx.fem.Function(C_space, dtype=np.float64)

num_cells_local = disconnected_mesh.topology.index_map(disconnected_mesh.topology.dim).size_local
indices = np.arange(num_cells_local, dtype=np.int32)
# To make sure that each matrix is assigned to the correct cell
midpoints: npt.NDArray[np.float_] = dolfinx.mesh.compute_midpoints(disconnected_mesh, disconnected_mesh.topology.dim, indices)
c_values = C_func.x.array.reshape((-1, N_max, T.shape[1]))
for local_idx, midpoint in enumerate(midpoints):
    #print(f"midpoint = {midpoint}")
    level, idx = hs.hmesh.find_active_cell(midpoint[:hs.dim])
    #print(f"midpoint = {midpoint}, level={level}, idx={idx}")
    mat: npt.NDArray = thb_operators[level, idx] @ hs.level_spaces[level].get_bezier_operator(idx).astype(np.float64)
    #mat = hs.local_multi_level_extraction_operator(idx, level, level) @ hs.level_spaces[level].get_bezier_operator(idx)
    # print(mat.shape)
    real_k, n_cols = mat.shape

    if real_k<N_max:
        padding_size = N_max - real_k
        
        #mat_padded = np.vstack((mat, np.zeros((padding_size, mat.shape[1])) ))
        Ci = mat#_padded
    else:
        padding_size=0
        Ci = mat
    
    # is a view of C_func.x.array, therefore we modify the content of C_func.x.array
    # No new array is created, the matrix->cell mapping is done here.
    c_values[local_idx, :, :] = np.vstack((Ci@T, np.zeros((padding_size, mat.shape[1]))))
C_func.x.scatter_forward()

In [ ]:
V_spline = create_spline_space(cells_to_dofs=padded_cells_to_dofs, mesh=disconnected_mesh, N_max=N_max, cell_type="hexahedron", dtype=dtype)
tabulate_A = make_bilinear_kernel(dtype, rtype, ufcx_kernel=kernela0, 
                                  padded_dofs=N_max, local_dofs=local_dofs_size)
tabulate_b = make_linear_kernel(dtype, rtype, ufcx_kernel=kernelf0,
                                padded_dofs=N_max, local_dofs=local_dofs_size)

In [ ]:
formtype = dolfinx.fem.form_cpp_class(dtype)  # type: ignore
# Gets the number of cells for which each individual core is responsible for.
cells = np.arange(msh.topology.index_map(msh.topology.dim).size_local, dtype=np.int32)

# The 4th argument np.array([...], dtype=np.int8) is the 
# active coefficients array. It lists which indices from the 
# coefficients list should be packed into the w_ pointer that the kernel receives.
integrals = {dolfinx.fem.IntegralType.cell: [
    (0, tabulate_A.address, cells, np.array([0], dtype=np.int8))]}

a_cond = dolfinx.fem.Form( # We are not forming anything yet, this is a recipe
    formtype( # selectes the correct floating-point precision
        spaces=[V_spline._cpp_object, 
                V_spline._cpp_object]
            , # trial and test spaces, determines the size of A_
        integrals=integrals, #this is a dictionary, and we are passing the adress of tabulate_A() here
        coefficients=[C_func._cpp_object
                    ], # weights w_, holds C@T
              constants=[],
              need_permutation_data=False,
              entity_maps=[], 
              mesh=msh._cpp_object)
)

integrals_rhs = {dolfinx.fem.IntegralType.cell: [(0, tabulate_b.address, cells, np.array([0], dtype=np.int8))]}
l_cond = dolfinx.fem.Form(
    formtype(
        spaces=[V_spline._cpp_object], # test space, determines the size of b_
        integrals=integrals_rhs, #give the adress of tabulate_b
        coefficients=[C_func._cpp_object], # holds the evaluations of f at the correct points, as well as C@T
        constants=[], need_permutation_data=False, entity_maps=[], mesh=msh._cpp_object
    )
)

In [ ]:
x_vec, A = solve_problem(hs=hs, a=a_cond, rhs=l_cond, dummy_index=np.max(padded_cells_to_dofs),
                         V_spline = V_spline, iterative=True, return_A=True, dirichlet_indices=None)

In [ ]:
# Extract the CSR (Compressed Sparse Row) arrays from PETSc
# indptr, indices, data = A.getValuesCSR()

# Get the global size of the matrix
# shape = A.getSize()

# # Create a SciPy CSR matrix
# A_scipy = sp.csr_array((data, indices, indptr), shape=shape)
# rows, cols = A_scipy.nonzero()
# with open("nnz_3d_hierarchical.dat", "w") as f_write:
#     for r, c in zip(rows, cols):
#         f_write.write(f"{c+1} {r+1}\n")
# # print(f"Matrix shape: {A_scipy.shape}")
# # print(f"Number of non-zeros: {A_scipy.nnz}")

# import matplotlib.pyplot as plt

# plt.figure(figsize=(7, 7))
# # plt.spy plots the non-zero entries of a matrix
# plt.spy(A_scipy, markersize=2, color='darkgray')
# plt.title("Sparsity Pattern of THB-Spline stiffness matrix \n Morton ordering")
# plt.show()

In [ ]:
cell_max = np.max(padded_cells_to_dofs, axis=1)
cell_min = np.min(padded_cells_to_dofs, axis=1)
cell_max = np.zeros(len(padded_cells_to_dofs), dtype=float)
cell_min = np.zeros_like(cell_max)
dummy_dof = np.max(padded_cells_to_dofs)
for i in range(len(cell_max)):
    cell_max[i] = np.max(padded_cells_to_dofs[i][padded_cells_to_dofs[i]!=dummy_dof])
    cell_min[i] = np.min(padded_cells_to_dofs[i][padded_cells_to_dofs[i]!=dummy_dof])
# 2. Compute the dilation per cell: χ(e) = max - min
dilation_per_cell = cell_max - cell_min

# 3. Compute global summary statistics
max_dilation = np.max(dilation_per_cell)
mean_dilation = np.mean(dilation_per_cell)
std_dilation = np.std(dilation_per_cell)
med_dilation = np.median(dilation_per_cell)

print("--- Dilation Metric Results ---")
print(f"Maximum Dilation (Worst-case): {max_dilation}")
print(f"Mean Dilation (Average element stretch): {mean_dilation:.2f}")
print(f"Standard Deviation of Dilation:     {std_dilation:.2f}")
print(f"Median Deviation of Dilation:     {med_dilation:.2f}")


In [ ]:
u_dg = dolfinx.fem.Function(V)

# Map the global B-spline coefficients back to local Legendre coefficients
for local_idx in range(num_cells_local):
    # Get global B-spline dof indices for this cell
    spline_dofs = padded_cells_to_dofs[local_idx]
    
    # Extract the B-spline coefficients for this cell
    u_spline_local = x_vec[spline_dofs]
    
    # Get the local transformation matrix G for this cell
    G = c_values[local_idx, :, :]
    
    # Transform B-spline to DG: mathematically, the kernel does A = G @ A0 @ G.T
    # This implies the coefficient mapping is u_dg = G.T @ u_spline
    u_dg_local = G.T @ u_spline_local
    
    # Assign to the standard DG function
    dg_dofs = V.dofmap.cell_dofs(local_idx)
    u_dg.x.array[dg_dofs] = u_dg_local

u_dg.x.scatter_forward()


# Compute exact L2 error using FEniCSx standard UFL
error_form = dolfinx.fem.form(ufl.inner(f - u_dg, f - u_dg) * dx_custom)
error_sq = dolfinx.fem.assemble_scalar(error_form)
exact_l2_error = np.sqrt(disconnected_mesh.comm.allreduce(error_sq, op=MPI.SUM))

print(f"Exact L2 Error (via DG projection): {exact_l2_error:.2e}")
rel_err = exact_l2_error/f_sq_integral
print(f"Relative error = {rel_err:.2e}")
# print(f"dofs = {A.getSize()}")

In [ ]:
print(f"({A.getSize()[0]}, {rel_err:.5e})")
A.destroy()

In [ ]:
V_error = dolfinx.fem.functionspace(disconnected_mesh, ("DG", 0))
v = ufl.TestFunction(V_error)

hQ = ufl.CellDiameter(disconnected_mesh)
volume_form = dolfinx.fem.form(1.0*v*dx_custom)
cell_volumes = dolfinx.fem.assemble_vector(volume_form).array

# Define the local L2 error form: integral of (f - u_dg)^2 per cell
# Note: We multiply by the test function 'v' to pick out each cell's contribution
local_error_form = dolfinx.fem.form(ufl.inner(f - u_dg, f-u_dg) * v * dx_custom)

err_cells = dorfler_marking(hierarchical_space=hs, theta=0.3, local_error_form=local_error_form)